# The World's Biggest Languages

Who are the top speakers on Earth, and what do macrolanguages mean for the count?
This notebook ranks global speaker totals from `low`, explores ISO scope codes, and
demonstrates the `LanguageCollection` lookup and filter API.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [2]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [3]:
db = low.LanguagesOfTheWorld()
print(db)
print(f"Total languages: {len(db.languages)}")
print(f"First ten: {[l.label for l in db.languages[:10]]}")

LanguagesOfTheWorld(languages=7929, countries=247, continents=5, scripts=106, speaker_counts=8969, language_names=167917)
Total languages: 7929
First ten: ["'Are'are", "'Auhelawa", "A'ou", 'A-Pucikwar', 'Aari', 'Aasáx', 'Abadi', 'Abaga', 'Abai Sungai', 'Abanyom']


## 2 · Polymorphic lookup

`db.languages.get()` accepts ISO 639-1 (2-char), ISO 639-3 (3-char), or a
case-insensitive label string.

In [4]:
for query in ("rw", "kin", "Kinyarwanda"):
    lang = db.languages.get(query)
    print(f"get({query!r}) → {lang.label} ({lang.part3}, ISO 639-1: {lang.part1})")

get('rw') → Kinyarwanda (kin, ISO 639-1: rw)
get('kin') → Kinyarwanda (kin, ISO 639-1: rw)
get('Kinyarwanda') → Kinyarwanda (kin, ISO 639-1: rw)


## 3 · Filter by minimum speakers

`filter()` returns languages whose global `speaker_count` meets the threshold.

In [5]:
popular = db.languages.filter(min_speakers=50_000_000)
print(f"Languages with ≥ 50 M speakers: {len(popular)}")

rows = [
    {
        "label": lang.label,
        "part3": lang.part3,
        "part1": lang.part1,
        "scope": lang.scope,
        "speaker_count": lang.speaker_count,
    }
    for lang in popular
]

df = pd.DataFrame(rows).sort_values("speaker_count", ascending=False)
df.head(15)

Languages with ≥ 50 M speakers: 42


,label,part3,part1,scope,speaker_count
4,Chinese,zho,zh,M,1299877520
21,Mandarin Chinese,cmn,NaN,I,1288700000
6,English,eng,en,I,1267445366
11,Hindi,hin,hi,I,544000000
29,Spanish,spa,es,I,485000000
0,Arabic,ara,ar,M,346277000
30,Standard Arabic,arb,NaN,I,335000000
1,Bengali,ben,bn,I,300000000
37,Urdu,urd,ur,I,288000000
26,Portuguese,por,pt,I,254300000


## 4 · Bar chart — top 30 languages (log scale)

In [6]:
top30 = df.head(30)

fig = px.bar(
    top30,
    x="speaker_count",
    y="label",
    orientation="h",
    color="scope",
    text="speaker_count",
    log_x=True,
    labels={"speaker_count": "Speakers (log scale)", "label": "", "scope": "ISO scope"},
    title="Top 30 Languages by Global Speaker Count",
    color_discrete_sequence=px.colors.qualitative.Set2,
)

fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=700, margin=dict(l=10, r=10, t=40, b=10))
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.show()

## 5 · Macrolanguages vs individual codes

ISO 639-3 scope `"M"` marks a macrolanguage that groups related individual codes.
Compare Chinese (`zho`) with its most-spoken members.

In [7]:
chinese = db.languages.get("zho")
print(f"{chinese.label}: scope={chinese.scope}, speakers={chinese.speaker_count:,}")

mandarin = db.languages.get("cmn")
cantonese = db.languages.get("yue")
for lang in (mandarin, cantonese):
    if lang:
        print(f"  {lang.label} ({lang.part3}): scope={lang.scope}, speakers={lang.speaker_count:,}")

Chinese: scope=M, speakers=1,299,877,520
  Mandarin Chinese (cmn): scope=I, speakers=1,288,700,000
  Yue Chinese (yue): scope=I, speakers=79,000,000


## 6 · Scope distribution

In [8]:
scope_labels = {"I": "Individual", "M": "Macrolanguage", "S": "Special"}

scope_df = pd.DataFrame([
    {"scope": scope_labels.get(lang.scope, lang.scope), "count": 1}
    for lang in db.languages
]).groupby("scope", as_index=False).sum()

fig2 = px.pie(
    scope_df,
    names="scope",
    values="count",
    title="ISO 639-3 Scope Distribution",
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig2.update_traces(textposition="inside", textinfo="percent+label")
fig2.show()

## 7 · Label substring filter

In [9]:
portuguese_variants = db.languages.filter(label_contains="Portug")
print(f"Languages with 'Portug' in the label: {len(portuguese_variants)}")
for lang in portuguese_variants[:8]:
    sc = f"{lang.speaker_count:,}" if lang.speaker_count else "—"
    print(f"  {lang.label} ({lang.part3}): {sc} speakers")

Languages with 'Portug' in the label: 5
  Indo-Portuguese (idb): 4,180 speakers
  Korlai Creole Portuguese (vkp): 800 speakers
  Malaccan Creole Portuguese (mcm): 2,200 speakers
  Portuguese (por): 254,300,000 speakers
  Portuguese Sign Language (psr): — speakers


## 8 · Summary

In [10]:
with_speakers = sum(1 for l in db.languages if l.speaker_count)
print(f"Languages with a global speaker count: {with_speakers:,} / {len(db.languages):,}")
print(f"Macrolanguages: {sum(1 for l in db.languages if l.scope == 'M')}")
print(f"Individual languages: {sum(1 for l in db.languages if l.scope == 'I')}")

Languages with a global speaker count: 5,858 / 7,929
Macrolanguages: 63
Individual languages: 7862
